In [1]:
" 2025-08-22"
"This program tests 4ul dispenses with 50ul filtered CORE tips"
"with Orange G into 200ul water in a Costar 96w round bottom plate. "
"The absorbance can be measured at 475 ± 10 nm (set filter wheel to 480 nm or closest)"

"DECK LAYOUT:"
"1. HTF tips at [0], 50UL tips at tip carrier [1]"
"2. COSTAR plate on carrier at rails 13 position [0]"
"3. Water trough behind COSTAR plate at position [1]"
"4. OrangeG dye 10mg/ml in trough on carrier at rails 19 position [0]"

"STEPS:"
"1. Rack HTF TIPS all channels. Aspirate 1000ul all channels."
"2. Dispense 200ul into first 4 columns. Repeat for next 2 sets of columns, 5-8, 9-12."
"3. Return HTF tips. Can be reused again."
"4. Rack 50ul tips. Aspirate 30ul OG, dispense 2ul back into trough for priming."
"5. Dispense 4ul into cols 1-6, all channels. Discard Tips."
"6. Repeat above, cols 7-11. New conditions?"
"7. Col 12 shall have controls: 4 blanks (water), 4 wells with 4ul dispensed with calibrated pipetter."
"7. Measure on plate reader."
"8. Tweak. Repeat."

'8. Tweak. Repeat.'

In [2]:
###############################################################################
# 0) SETUP (unchanged)
###############################################################################
%load_ext autoreload
%autoreload 2
import asyncio

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import (
    STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
)
from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped
from pylabrobot.resources.agenbio.plates import AGenBio_1_troughplate_100000uL_Fl
# from pylabrobot.resources.corning.plates import Cor_96_wellplate_360ul_Fb
from pylabrobot.resources.celltreat.plates import CellTreat_96_wellplate_350ul_Ub
from pylabrobot.resources import (
    hamilton_96_tiprack_1000uL_filter,             # 1000 µL filtered tips
    hamilton_96_tiprack_50uL_filter,        # 50 µL filtered tips
    hamilton_96_tiprack_10uL_filter              # 10 µL filtered tips
)


In [3]:

###############################################################################
# 1) BUILD LH + DECK
###############################################################################
backend = STARBackend()
lh = LiquidHandler(backend=backend, deck=STARLetDeck())
await lh.setup(skip_autoload=True)        # faster while iterating

In [4]:

##############################################################################
#2) CARRIERS & LABWARE
##############################################################################

# # tips
tip_car        = TIP_CAR_480_A00("tip_car")
tip_car[0]     = hamilton_96_tiprack_1000uL_filter(name="htf_tips")       # 300 µL
tip_car[1]     = hamilton_96_tiprack_50uL_filter(name="htf_50ul")  # 50 µL
tip_car[2]     = hamilton_96_tiprack_10uL_filter(name="htf_10ul")  # 10 µL
lh.deck.assign_child_resource(tip_car, rails=25)
htf_tips  = lh.deck.get_resource("htf_tips")   # 1000 µL filtered rack
tips_50ul = lh.deck.get_resource("htf_50ul")   # 50 µL filtered rack
tips_10ul = lh.deck.get_resource("htf_10ul")   # 10 µL filtered rack

# --- SOURCE CARRIER (MFX) --- orange G
src_mod  = Hamilton_MFX_plateholder_DWP_metal_tapped("src_mod")
src_car  = MFX_CAR_L5_base("src_car", modules={0: src_mod})
lh.deck.assign_child_resource(src_car, rails=19)

src_plate = AGenBio_1_troughplate_100000uL_Fl("source_plate")
src_mod.assign_child_resource(src_plate)

# --- DESTINATION + WATER (MFX) ---
dst_mod   = Hamilton_MFX_plateholder_DWP_metal_tapped("dst_mod")
water_mod = Hamilton_MFX_plateholder_DWP_metal_tapped("water_mod")

dst_car = MFX_CAR_L5_base("dst_car", modules={0: dst_mod, 1: water_mod})
lh.deck.assign_child_resource(dst_car, rails=13)

dst_plate   = CellTreat_96_wellplate_350ul_Ub("dest_plate")
water_plate = AGenBio_1_troughplate_100000uL_Fl("water_plate")

dst_mod.assign_child_resource(dst_plate)
water_mod.assign_child_resource(water_plate)


In [5]:
CHANNELS_8 = list(range(8))
ROWS = "ABCDEFGH"

def col_wells(plate, col: int):
    return [plate[f"{r}{col}"][0] for r in ROWS]

async def fill_plate_with_water():
    # 1) Rack 1000 µL filtered tips on all channels
    await lh.pick_up_tips(htf_tips["A1:H1"], use_channels=CHANNELS_8)

    precise_trough = dict(
        lld_mode=[STARBackend.LLDMode.GAMMA]*8,
        gamma_lld_sensitivity=[2]*8,
        immersion_depth=[1]*8,
        immersion_depth_direction=[0]*8,
        surface_following_distance=[1]*8,
        transport_air_volume=[0]*8,
        settling_time=[1]*8
    )

    # 2-5) Fill 4 columns at a time (1-4, 5-8, 9-12)
    for block_start in (1, 5, 9):
        block_cols = range(block_start, block_start + 4)

        await lh.aspirate(
            water_plate["A1"]*8,
            vols=[950]*8,
            use_channels=CHANNELS_8,
            **precise_trough
        )
        # prime
        await lh.dispense(
            water_plate["A1"]*8,
            vols=[100]*8,
            use_channels=CHANNELS_8,
            **precise_trough
        )

        for c in block_cols:
            await lh.dispense(
                col_wells(dst_plate, c),
                vols=[200]*8,
                use_channels=CHANNELS_8,
                liquid_height=[5]*8,
                transport_air_volume=[0]*8,
                settling_time=[1]*8,
                flow_rates=[60]*8
            )

        # clear remainder to trough
        await lh.dispense(
            water_plate["A1"]*8,
            vols=[0]*8,
            use_channels=CHANNELS_8,
            **precise_trough,
            blow_out=[True]*8
        )

    # 6) Discard tips
    await lh.discard_tips()


In [9]:
# ---- PRECISE DISPENSE ----
precise_dispense = dict(
    lld_mode=[STARBackend.LLDMode.GAMMA]*8,
    gamma_lld_sensitivity=[2]*8,
    immersion_depth=[0]*8,
    immersion_depth_direction=[0]*8,
    transport_air_volume=[0]*8,
    flow_rates=[10]*8,
    swap_speed=[60]*8,
    settling_time=[1]*8,
    jet=[False]*8,
    blow_out=[False]*8
)

# ---- ASPIRATION (Orange G source) ----
aspirate_OG = dict(
    lld_mode=[STARBackend.LLDMode.GAMMA]*8,
    gamma_lld_sensitivity=[2]*8,
    immersion_depth=[1]*8,
    immersion_depth_direction=[0]*8,
    flow_rates=[12]*8,
    settling_time=[1]*8,
    # mix_volume=[10]*8,
    # mix_cycles=[2]*8,
    # mix_speed=[10]*8,
    transport_air_volume=[0]*8
)

async def add_orangeG_10ul_tips():
    # 1) Rack 10 µL filtered tips on all channels
    await lh.pick_up_tips(tips_10ul["A1:H1"], use_channels=CHANNELS_8)

    for c in range(1, 6):
        # 2) Aspirate 10 µL from Orange G trough
        await lh.aspirate(src_plate["A1"]*8, vols=[10]*8, use_channels=CHANNELS_8, **aspirate_OG)
        # 3) Prime 4 µL back into trough (no air gap)
        await lh.dispense(src_plate["A1"]*8, vols=[4]*8, use_channels=CHANNELS_8, lld_mode=[STARBackend.LLDMode.GAMMA]*8)
        # 4) Dispense 4 µL into column
        await lh.dispense(
            col_wells(dst_plate, c),
            vols=[4]*8,
            use_channels=CHANNELS_8,
            liquid_height=[4]*8,
            **precise_dispense
        )
        # 5) Dispense remaining 2 µL back into trough with blowout
        await lh.dispense(
            src_plate["A1"]*8,
            vols=[2]*8,
            use_channels=CHANNELS_8,
            lld_mode=[STARBackend.LLDMode.GAMMA]*8,
            blow_out=[True]*8,
            empty=[True]*8
        )

    # 7) Discard tips
    # await lh.discard_tips()
    await lh.drop_tips(tips_10ul["A1:H1"], use_channels=CHANNELS_8)


In [ ]:
async def add_orangeG_50ul_tips():
    # 1) Rack 50 µL filtered tips on all channels
    await lh.pick_up_tips(tips_50ul["A1:H1"], use_channels=CHANNELS_8)

    for c in range(4,9):
        # 2) Aspirate 20 µL from Orange G trough
        await lh.aspirate(src_plate["A1"]*8, vols=[20]*8, use_channels=CHANNELS_8, **aspirate_OG)
        # 3) Prime 14 µL back into trough (no air gap)
        await lh.dispense(src_plate["A1"]*8, vols=[14]*8, use_channels=CHANNELS_8, lld_mode=[STARBackend.LLDMode.GAMMA]*8)
        # 4) Dispense 4 µL into column
        await lh.dispense(
            col_wells(dst_plate, c),
            vols=[4]*8,
            use_channels=CHANNELS_8,
            liquid_height=[4]*8,
            **precise_dispense
        )
        # 5) Dispense remaining 2 µL back into trough with blowout
        await lh.dispense(
            src_plate["A1"]*8,
            vols=[2]*8,
            use_channels=CHANNELS_8,


            
            lld_mode=[STARBackend.LLDMode.GAMMA]*8,
            blow_out=[True]*8,
            empty=[True]*8
        )

    # 7) Discard tips
    await lh.discard_tips()


In [10]:
# await fill_plate_with_water()
await add_orangeG_10ul_tips()
# await add_orangeG_50ul_tips()

In [ ]:
# TROUBLESHOOTING CODE SNIPPETS
# await lh.dispense(src, vols=[50]*8, use_channels=[0], blow_out=[1])
# await lh.blow_out(location=waste["A1"], use_channels=channels)
# await lh.dispense(src_wells, vols=[100]*8, use_channels=USE_CHANS, **precise, liquid_height=[30]*8)   # no blow-out
# # tip_rack = lh.deck.get_resource("htf_tips")
# await lh.drop_tips(htf_tips["A1:H1"], use_channels=list(range(8)))
# await lh.dispense(water_plate["A1"]*8, vols=[10]*8, use_channels=CHANNELS_8, lld_mode=[STARBackend.LLDMode.GAMMA]*8)
# await lh.dispense(src_plate["A1"]*8, vols=[10]*8, use_channels=CHANNELS_8, lld_mode=[STARBackend.LLDMode.GAMMA]*8, blow_out=[True]*8)
# # await lh.drop_tips(tiprack["A1:C1"])
# await lh.drop_tips(tips_50ul["A1:H1"], use_channels=CHANNELS_8)  # 50 µL filtered tips

await lh.drop_tips(tips_10ul["A1:H1"], use_channels=CHANNELS_8)
# await lh.drop_tips(tips_10ul["A1:F1"], use_channels=[0,1,2,3,4,5])
# await lh.drop_tips(tips_10ul["H1"], use_channels=[7])

# await lh.discard_tips()
# # await lh.prepare_for_manual_channel_operation()
# await lh.stop()